In [ ]:
import jax
import jax.numpy as jnp
from jax import lax

# Dimensionless kernel prefactor (-1.0 / ( 4.0 * (jnp.pi**3/2) * jnp.sqrt(1j) ) )
def solve_eta(scattering_length, time_interval=1.0, kernel_prefactor=(-1.0/(4.0*(jnp.pi**3/2)*jnp.sqrt(1j))) ):
    scattering_length = jnp.asarray(scattering_length, dtype=jnp.complex64)

    num_points = scattering_length.shape[0]
    num_steps = num_points - 1
    dt = time_interval / num_steps

    eta_0 = -4.0 * jnp.pi * scattering_length[0]
    eta = jnp.zeros_like(scattering_length, dtype=jnp.complex64).at[0].set(eta_0)

    l1_prefactor = 2.0 * kernel_prefactor / jnp.sqrt(dt)
    j = jnp.arange(num_steps)

    def time_step(history, k):
        diffs = history[1:] - history[:-1]

        # Only j = 0, ..., k - 2 is known. The j = k - 1 term contains eta[k]
        # and is moved into the denominator below.
        valid = j < (k - 1)
        m = k - j
        safe_m = jnp.maximum(m, 1)
        weights = jnp.where(
            valid,
            jnp.sqrt(safe_m) - jnp.sqrt(safe_m - 1),
            0.0,
        )

        known_history = jnp.sum(weights * diffs)
        numerator = -1.0 + l1_prefactor * (history[k - 1] - known_history)
        denominator = 1.0 / (4.0 * jnp.pi * scattering_length[k]) + l1_prefactor
        eta_k = numerator / denominator

        history = history.at[k].set(eta_k)
        return history, eta_k

    indices = jnp.arange(1, num_points)
    eta, _ = lax.scan(time_step, eta, indices)

    return eta

def lossless_contact(non_dim_eta):
    return jnp.abs(non_dim_eta)**2

def diff(grid):
    return grid[1:]-grid[:-1]

def cumulative_trapez_int(integrand, dx):
    interval_integrals = 0.5 * dx * (integrand[:-1] + integrand[1:])
    initial_value = jnp.zeros((1,), dtype=interval_integrals.dtype)
    return jnp.concatenate((initial_value, jnp.cumsum(interval_integrals)))

def energy_density(scattering_length, eta):
    # for elastic models with real scattering lengths. 
    a_s = jnp.asarray(scattering_length, dtype=jnp.float32) 
    eta = jnp.asarray(eta, dtype=jnp.complex64) 
    
    prefix = 1/(8*jnp.pi)

    integr = lossless_contact(eta) / (a_s**2)
    ener_den = prefix * cumulative_trapez_int(integr, diff(a_s))

    return ener_den